# Using the Jupyter Notebook Terminal
You have the choice between using the Jupyter Notebook cells to compile, execute and visualise your model or to use the terminal. In this chapter we will explain how to use the terminal and how to use the needed commands and repository.

## First time use


1. Choose correct kernel.

runtime > change runtime > Python3; T4 GPU; newest > save

2. Open terminal in Jupyter Notebook.


3. Mount Google Drive to use your Drive-files in the Jupyter-Notebook enviroment.
For that execute the following code cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


4. Open terminal in Jupyter Notebook.
file > New > Terminal


5. Change directory to your drive file (usually "mydrive")


```
cd drive/MyDrive
```
6. Install vedo k3d (evertime)


```
pip install vedo k3d
pip install imageio
```
7. Run this cell everytime, so you can use vedo in the Colab Enviroment.



In [ ]:
import vedo
vedo.settings.init_colab()

setting up colab environment (can take a minute) ...installing k3d... setup completed.


7. Import the repository using
```
git clone https://github.com/mariecvk/yallamain.git
```
8. Use the "ls" command to check if the repository is installed. It will list all the files in the current directory.

```
ls
```

10. Change directory to yallamain.


```
cd yallamain/analyse
```

11. Create your foundation.cu file. This code is crucial for the simulation because it contains the information what you want to simulate.

Either change the code in the cell and execute cell to overwrite foundation.cu (has to be manually uploaded first because is not included in repository) or
copy the code of the cell below and paste it into your chosen IDE and change it.

12. Upload your foundation.cu file into the directory, name it whatever you want it to be named. Important is, that it's still .cu file.
This has to be done manually. Make sure its in the correct file (yallamain).

13. Check if the upload was successful. And go back to the "yallamain" directory.

```
cd ..
```


14. Now to the fun part. Compile, execute and visualise your simualtion.
Use this command:


```
python script.py foundation.cu
```

Make sure to change the filename, if you didn't use 'foundation' as filename.

15. Download the created video, which is located in

`mydrive/yallamain/yalla_runs/run_XXX/output`

There is a chance, that the Quicktimeplayer won't be able to play the video of the simulation, if so download VLC media player, then it should work.

15. To visualise the simulation with the browser version change directory to the complementary directory e.g.:

```
cd yalla_runs/run_016/output
```


```
python ../../../visualize_vtk.py example_01.vtk
```
Change example_01.vtk with the correct file name.



## Every other use

1. Choose correct kernel.

runtime > change runtime > Python3; T4 GPU; newest > save

2. Open terminal in Jupyter Notebook.

3. Change to yallamain directory.


```
cd drive/MyDrive/yallamain
```
4. Install Vedo.

```
pip install vedo
```
5. Execute filename.cu. Uploade the files into the analyse file or use previously uploaded files from examples or analyse.


```
python script.py filename.cu
```
6. Download your output from

`yallamain/yalla_runs/run_XXX/output`

7. To visualise the simulation with the browser version change directory to the complementary directory e.g.:

`cd yalla_runs/run_016/output`


`python ../../../visualize_vtk.py example_01.vtk`


Change example_01.vtk with the correct file name.




To change the parameters in your filename.cu, you can either paste the file into a cell, edit the parameters there, and execute the cell. Alternatively, you can double-click the file in your drive within the Jupyter Notebook environment to open it and modify the parameters directly. After saving the changes, wait a few seconds before running the command.

In [ ]:
%%writefile filename.cu

#include "../include/dtypes.cuh"
#include "../include/inits.cuh"
#include "../include/property.cuh"
#include "../include/solvers.cuh"
#include "../include/vtk.cuh"
#include <cmath>


// cell-cell interaction parameter
const auto r_max = 1.f;  // Max Radius where cells interact with each other
const auto r_min = 0.5f;  // the force is 0 for cells with this distance -> equilibrium distance

// simulation parameter
const auto n_time_steps = 10u;
const auto dt = 0.05f;  // size of the timesteps -> smaller = more detailed and slower

// for creating the cell sheet we define the number of cells in a row (=> n_circumference) and the number of layers (=> n_layer)
const int n_circumference = 12;
const int n_layer = 5;
const auto n_cells = n_circumference * n_layer;


// Parameter for cylinder
const auto R_zylinder = n_circumference * r_min / (2.f * (float)M_PI);   // calculating radius of cylinder by circumference (n_circumference * r_min) divided by PI
const auto k_radial = 1.0f;  // higher number = cells stay better at clyinder shape


// counter for the size of the neighborhood of a cell
__device__ int* d_neig;

// R_zylinder als device-Konstante verfügbar machen
__device__ float  d_R_dev;

// simulation_step is called for every pair of cells (i,j) and calculates a vector of forces, which moves cell i
// Xi = position of cell i
// r = vector from j to i
// dist = distance between i and j
// i = index of i
// j = index of j
// calculating a force away from j = r * F
__device__ float3 simulation_step(
    float3 Xi, float3 r, float dist, int i, int j)
{
    // The forces are calaculated for cell tupels that are not the same and where distance is not too big
    float3 dF{0};


    if (i == j) {
        // calculating the ditance from cell i to the z-axis (the z_axis is in the middel of the cylinder)
        float distance_i_to_zAxis = sqrtf(Xi.x * Xi.x + Xi.y * Xi.y);

        if (distance_i_to_zAxis > 0.f) {
            // calculating the vector that describes the direction of cell i with origin in z-axis
            // because the vector is normed to 1 -> devided by distance_i_to_zAxis
            float3 vector_direction_i = {Xi.x / distance_i_to_zAxis, Xi.y / distance_i_to_zAxis, 0.f};

            // dF describes the force of the cylinder
            // positive = force outward, negative = force inside
            float deviation = R_zylinder - distance_i_to_zAxis;
            // deviation descirbes the difference between the radius of the cylinder (this i wehere the cells should be positioned) and the actual distance from the middel of the cylinder to cell i
            dF = vector_direction_i * (k_radial * deviation);
            // dF calculates if the cell i has to move outward or inside (or stay) to be on place with the radius of the cylinder
        }
        return dF;
    }


    if (dist > r_max) return dF;  // if to cells i and j are to fr away from each other -> no forces acting on each other

    d_neig[i] += 1;  // counting +1 for neigborhood for every cell j that interacts with i


    // forces between the cell pair i and j
    // -> when to cells are to close = repulsion
    // -> when to cells are to far away = attraction
    auto a = 1.f;
    auto F = a * (1.f - 2.f * dist);
    dF = r * F / dist;
    return dF;
}

int main(int argc, const char* argv[])
{

    cudaMemcpyToSymbol(d_R_dev, &R_zylinder, sizeof(float));

    Solution<float3, Gabriel_solver> cells{n_cells, 50, r_max};

    // initialization of the cylinder
    float dz       = r_min * sqrtf(3.f) / 2.f;   // dz = distance between two layers of the cylinder
    float d_angle = 2.f * (float)M_PI / (float)n_circumference;       // calculating the regular angle for the number of cells in a layer (=n_circumference) ex. 360degree/12 = 30 degree

    // looping through each layer
    for (int s = 0; s < n_layer; s++) {
      // looping through each cell at the layer
        for (int p = 0; p < n_circumference; p++) {
            int   idx    = s * n_circumference + p;  // define an index for every cell ex. layer 2, cell 4 = 2*60+4 = 124
            float angle = p * d_angle;  // calculating the angle for each cell -> 0, 30, 60, 90, 120, .., 330

            // for every second layer, the cells are movet slightly to the right to create an hexagon shape
            if (s % 2 == 1) angle += d_angle * 0.5f;

            // every cell with index idx gets their position at the cylinder -> using the radius and cos/sin to caldulate coordinates
            cells.h_X[idx] = {
                R_zylinder * cosf(angle),   // x-coordinate
                R_zylinder * sinf(angle),   // y-coordinate
                s * dz                        // z-coordinate
            };
        }
    }

    cells.copy_to_device();

    Property<int> h_neig{n_cells, "neighbours"};
    cudaMemcpyToSymbol(d_neig, &h_neig.d_prop, sizeof(d_neig));

    auto fun = [&](const int n, const float3* __restrict__ d_X, float3* d_dX) {
        thrust::fill(thrust::device, h_neig.d_prop, h_neig.d_prop + n, 0);
    };


    Vtk_output output{"cylinder"};
    // in every time step the simulation_step function is called one time
    for (auto time_step = 0; time_step <= n_time_steps; time_step++) {
        cells.copy_to_host();
        h_neig.copy_to_host();
        cells.take_step<simulation_step>(dt, fun);
        output.write_positions(cells);
        output.write_property(h_neig);
    }
    return 0;
}

# Using Jupyter Notebook Cells

1. Mount Google Drive to Jupyter Notebook.

In [ ]:
!cd /content

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

2. Change directory to your drive.
Clone repository into drive.

In [ ]:
!cd /content/drive/MyDrive/yallamain
!git clone https://github.com/mariecvk/yallamain.git

3. Check if the cloning was successful.

In [ ]:
!ls
# You should see analyse examples include tests LICENSE
# README.md branching.gif create_video_script.py
# script.py visualize_vtk.py

4. Install vedo for the Google Colab enviroment.

In [ ]:
!pip install vedo k3d
import vedo
vedo.settings.init_colab()

5. Simulate your model. Adjust the code below.

In [ ]:
%%writefile filename.cu

#include "../include/dtypes.cuh"
#include "../include/inits.cuh"
#include "../include/property.cuh"
#include "../include/solvers.cuh"
#include "../include/vtk.cuh"
#include <cmath>


// cell-cell interaction parameter
const auto r_max = 1.f;  // Max Radius where cells interact with each other
const auto r_min = 0.5f;  // the force is 0 for cells with this distance -> equilibrium distance

// simulation parameter
const auto n_time_steps = 10u;
const auto dt = 0.05f;  // size of the timesteps -> smaller = more detailed and slower

// for creating the cell sheet we define the number of cells in a row (=> n_circumference) and the number of layers (=> n_layer)
const int n_circumference = 12;
const int n_layer = 5;
const auto n_cells = n_circumference * n_layer;


// Parameter for cylinder
const auto R_zylinder = n_circumference * r_min / (2.f * (float)M_PI);   // calculating radius of cylinder by circumference (n_circumference * r_min) divided by PI
const auto k_radial = 1.0f;  // higher number = cells stay better at clyinder shape


// counter for the size of the neighborhood of a cell
__device__ int* d_neig;

// R_zylinder als device-Konstante verfügbar machen
__device__ float  d_R_dev;

// simulation_step is called for every pair of cells (i,j) and calculates a vector of forces, which moves cell i
// Xi = position of cell i
// r = vector from j to i
// dist = distance between i and j
// i = index of i
// j = index of j
// calculating a force away from j = r * F
__device__ float3 simulation_step(
    float3 Xi, float3 r, float dist, int i, int j)
{
    // The forces are calaculated for cell tupels that are not the same and where distance is not too big
    float3 dF{0};


    if (i == j) {
        // calculating the ditance from cell i to the z-axis (the z_axis is in the middel of the cylinder)
        float distance_i_to_zAxis = sqrtf(Xi.x * Xi.x + Xi.y * Xi.y);

        if (distance_i_to_zAxis > 0.f) {
            // calculating the vector that describes the direction of cell i with origin in z-axis
            // because the vector is normed to 1 -> devided by distance_i_to_zAxis
            float3 vector_direction_i = {Xi.x / distance_i_to_zAxis, Xi.y / distance_i_to_zAxis, 0.f};

            // dF describes the force of the cylinder
            // positive = force outward, negative = force inside
            float deviation = R_zylinder - distance_i_to_zAxis;
            // deviation descirbes the difference between the radius of the cylinder (this i wehere the cells should be positioned) and the actual distance from the middel of the cylinder to cell i
            dF = vector_direction_i * (k_radial * deviation);
            // dF calculates if the cell i has to move outward or inside (or stay) to be on place with the radius of the cylinder
        }
        return dF;
    }


    if (dist > r_max) return dF;  // if to cells i and j are to fr away from each other -> no forces acting on each other

    d_neig[i] += 1;  // counting +1 for neigborhood for every cell j that interacts with i


    // forces between the cell pair i and j
    // -> when to cells are to close = repulsion
    // -> when to cells are to far away = attraction
    auto a = 1.f;
    auto F = a * (1.f - 2.f * dist);
    dF = r * F / dist;
    return dF;
}

int main(int argc, const char* argv[])
{

    cudaMemcpyToSymbol(d_R_dev, &R_zylinder, sizeof(float));

    Solution<float3, Gabriel_solver> cells{n_cells, 50, r_max};

    // initialization of the cylinder
    float dz       = r_min * sqrtf(3.f) / 2.f;   // dz = distance between two layers of the cylinder
    float d_angle = 2.f * (float)M_PI / (float)n_circumference;       // calculating the regular angle for the number of cells in a layer (=n_circumference) ex. 360degree/12 = 30 degree

    // looping through each layer
    for (int s = 0; s < n_layer; s++) {
      // looping through each cell at the layer
        for (int p = 0; p < n_circumference; p++) {
            int   idx    = s * n_circumference + p;  // define an index for every cell ex. layer 2, cell 4 = 2*60+4 = 124
            float angle = p * d_angle;  // calculating the angle for each cell -> 0, 30, 60, 90, 120, .., 330

            // for every second layer, the cells are movet slightly to the right to create an hexagon shape
            if (s % 2 == 1) angle += d_angle * 0.5f;

            // every cell with index idx gets their position at the cylinder -> using the radius and cos/sin to caldulate coordinates
            cells.h_X[idx] = {
                R_zylinder * cosf(angle),   // x-coordinate
                R_zylinder * sinf(angle),   // y-coordinate
                s * dz                        // z-coordinate
            };
        }
    }

    cells.copy_to_device();

    Property<int> h_neig{n_cells, "neighbours"};
    cudaMemcpyToSymbol(d_neig, &h_neig.d_prop, sizeof(d_neig));

    auto fun = [&](const int n, const float3* __restrict__ d_X, float3* d_dX) {
        thrust::fill(thrust::device, h_neig.d_prop, h_neig.d_prop + n, 0);
    };


    Vtk_output output{"cylinder"};
    // in every time step the simulation_step function is called one time
    for (auto time_step = 0; time_step <= n_time_steps; time_step++) {
        cells.copy_to_host();
        h_neig.copy_to_host();
        cells.take_step<simulation_step>(dt, fun);
        output.write_positions(cells);
        output.write_property(h_neig);
    }
    return 0;
}

6. Compile and execute the code. You can always use the codes germanapp or we prepared for you in analyse or examples by changing foundation.cu foundation to the needed filename.

In [ ]:
!nvcc -std=c++11 -arch=sm_75 foundation.cu -o foundation
!./foundation

7. Create the visualisation.

In [ ]:
import vedo
import glob, os, re
import imageio.v2 as imageio

# load VTK files and sort them
vtk_files = sorted(
    glob.glob("/content/drive/MyDrive/yallamain/yalla_runs/run_000/output/*.vtk"),
    key=lambda x: int(re.search(r"cylinder_(\d+)\.vtk$", x).group(1))
)
print(f"{len(vtk_files)} VTK-Files found")

os.makedirs("/content/drive/MyDrive/yallamain/frames", exist_ok=True)

plt = vedo.Plotter(offscreen=True, size=(800, 600), bg="white")

plt.renderer.SetBackground(1, 1, 1)

frame_paths = []

# calculating how big the sheet is for positioning of the view
first_pts = vedo.Points(vtk_files[0])
pos = first_pts.points
cx = (pos[:, 0].max() + pos[:, 0].min()) / 2
cy = (pos[:, 1].max() + pos[:, 1].min()) / 2
spread = max(pos[:, 0].max() - pos[:, 0].min(),
             pos[:, 1].max() - pos[:, 1].min())
cam_z = spread * 4  # how far away the camera is positioned

for i, f in enumerate(vtk_files):
    pts = vedo.Points(f)
    positions = pts.points

    spheres = vedo.Spheres(
        positions,
        r=0.25,
        res=12,
        c="lightblue",
    )
    spheres.lighting("glossy")

    plt.clear()
    plt.show(
        spheres,
        resetcam=(i == 0),
        viewup="y",
        camera={
            "pos": [cx, cy, cam_z],
            "focalPoint": [cx, cy, 0],
            "viewup":     [0, 1, 0]
            },
    )

    frame_path = f"/content/drive/MyDrive/yallamain/frames/frame_{i:04d}.png"
    plt.screenshot(frame_path)
    frame_paths.append(frame_path)

plt.close()
print(f"{len(frame_paths)} Frames gerendert")

# Frames to MP4 file
video_path = "/content/drive/MyDrive/yallamain/yalla_runs/run_000/output/cell_simulation.mp4"

with imageio.get_writer(video_path, fps=10, macro_block_size=1) as writer:
    for p in frame_paths:
        writer.append_data(imageio.imread(p))

print("Video saved:", video_path)


8. Download your simulation video manually from your drive.